In [15]:
import pandas as pd
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.metrics import confusion_matrix

df = pd.read_csv('KO_weekly_return_volatility.csv')
daily_data = pd.read_csv('KO_weekly_return_detailed.csv')

train = df[df['Year'].isin([2021, 2022, 2023])]
test = df[df['Year'].isin([2024, 2025])]
daily_test = daily_data[daily_data['Year'].isin([2024, 2025])].reset_index(drop=True)

X_train = train[['mean_return', 'volatility']].values
Y_train = train['decision'].values
X_test = test[['mean_return', 'volatility']].values
Y_test = test['decision'].values

print(X_train.shape)

lda = LinearDiscriminantAnalysis()
qda = QuadraticDiscriminantAnalysis(store_covariance=True)
lda.fit(X_train, Y_train)
qda.fit(X_train, Y_train)

(156, 2)


,"solver solver: {'svd', 'eigen'}, default='svd'Solver to use, possible values: - 'svd': Singular value decomposition (default). Does not compute the covariance matrix, therefore this solver is recommended for data with a large number of features. - 'eigen': Eigenvalue decomposition. Can be combined with shrinkage or custom covariance estimator.",'svd'
,"shrinkage shrinkage: 'auto' or float, default=NoneShrinkage parameter, possible values: - None: no shrinkage (default). - 'auto': automatic shrinkage using the Ledoit-Wolf lemma. - float between 0 and 1: fixed shrinkage parameter. Enabling shrinkage is expected to improve the model when some classes have a relatively small number of training data points compared to the number of features by mitigating overfitting during the covariance estimation step.This should be left to `None` if `covariance_estimator` is used.Note that shrinkage works only with 'eigen' solver.",None
,"priors priors: array-like of shape (n_classes,), default=NoneClass priors. By default, the class proportions are inferred from thetraining data.",None
,"reg_param reg_param: float, default=0.0Regularizes the per-class covariance estimates by transforming S2 as``S2 = (1 - reg_param) * S2 + reg_param * np.eye(n_features)``,where S2 corresponds to the `scaling_` attribute of a given class.",0.0
,"store_covariance store_covariance: bool, default=FalseIf True, the class covariance matrices are explicitly computed andstored in the `self.covariance_` attribute... versionadded:: 0.17",True
,"tol tol: float, default=1.0e-4Absolute threshold for the covariance matrix to be considered rankdeficient after applying some regularization (see `reg_param`) to each`Sk` where `Sk` represents covariance matrix for k-th class. Thisparameter does not affect the predictions. It controls when a warningis raised if the covariance matrix is not full rank... versionadded:: 0.17",0.0001
,"covariance_estimator covariance_estimator: covariance estimator, default=NoneIf not None, `covariance_estimator` is used to estimate the covariancematrices instead of relying on the empirical covariance estimator(with potential shrinkage). The object should have a fit method anda ``covariance_`` attribute like the estimators in:mod:`sklearn.covariance`. If None the shrinkage parameter drives theestimate.This should be left to `None` if `shrinkage` is used.Note that `covariance_estimator` works only with the 'eigen' solver.",None


## Q1 : What is the equation for the linear and quadratic classifier found from training data?

In [16]:
print(f'LDA: {lda.coef_[0][0]:.4f}*mean_return + {lda.coef_[0][1]:.4f}*volatility + {lda.intercept_[0]:.4f} = 0')

print('\nQDA:')
for i, cls in enumerate(qda.classes_):
    print(f'\nClass {cls}:')
    print(f'  mean = {qda.means_[i]}')
    print(f'  prior = {qda.priors_[i]:.4f}')
    print(f'  covariance matrix =')
    print(f'  {qda.covariance_[i]}')
    cov_inv = np.linalg.inv(qda.covariance_[i])
    log_det = np.log(np.linalg.det(qda.covariance_[i]))
    print(f'  Quadratic form: (x - mu)^T * Sigma^-1 * (x - mu) + ln|Sigma| - 2*ln(prior)')
    print(f'  ln|Sigma| = {log_det:.4f}')
    print(f'  -2*ln(prior) = {-2*np.log(qda.priors_[i]):.4f}')

LDA: 0.8368*mean_return + 0.3385*volatility + -0.9227 = 0

QDA:

Class NO:
  mean = [-0.02960941  0.78544514]
  prior = 0.6474
  covariance matrix =
  [[ 0.23531025 -0.05996793]
 [-0.05996793  0.1841742 ]]
  Quadratic form: (x - mu)^T * Sigma^-1 * (x - mu) + ln|Sigma| - 2*ln(prior)
  ln|Sigma| = -3.2253
  -2*ln(prior) = 0.8695

Class YES:
  mean = [0.14147    0.79866132]
  prior = 0.3526
  covariance matrix =
  [[ 0.2169197  -0.06104397]
 [-0.06104397  0.19567693]]
  Quadratic form: (x - mu)^T * Sigma^-1 * (x - mu) + ln|Sigma| - 2*ln(prior)
  ln|Sigma| = -3.2514
  -2*ln(prior) = 2.0850


## Q2 : What is the accuracy for testing years for each classifier? Which classifier is better?

In [8]:
lda_preds = lda.predict(X_test)
qda_preds = qda.predict(X_test)
lda_acc = round(lda.score(X_test, Y_test) * 100, 2)
qda_acc = round(qda.score(X_test, Y_test) * 100, 2)
print(f'LDA accuracy = {lda_acc}%, QDA accuracy = {qda_acc}%')
print(f'Better classifier: {"LDA" if lda_acc >= qda_acc else "QDA"}')

LDA accuracy = 67.92%, QDA accuracy = 70.75%
Better classifier: QDA


## Q3 : Confusion matrix for testing period for each classifier

In [9]:
cm_lda = confusion_matrix(Y_test, lda_preds, labels=['YES', 'NO'])
cm_qda = confusion_matrix(Y_test, qda_preds, labels=['YES', 'NO'])
print('LDA Confusion Matrix:')
print(f'  Predicted YES  Predicted NO')
print(f'Actual YES  {cm_lda[0][0]}           {cm_lda[0][1]}')
print(f'Actual NO   {cm_lda[1][0]}           {cm_lda[1][1]}')
print('\nQDA Confusion Matrix:')
print(f'  Predicted YES  Predicted NO')
print(f'Actual YES  {cm_qda[0][0]}           {cm_qda[0][1]}')
print(f'Actual NO   {cm_qda[1][0]}           {cm_qda[1][1]}')

LDA Confusion Matrix:
  Predicted YES  Predicted NO
Actual YES  3           30
Actual NO   4           69

QDA Confusion Matrix:
  Predicted YES  Predicted NO
Actual YES  4           29
Actual NO   2           71


## Q4 : True positive rate (sensitivity) and true negative rate (specificity) for testing years

In [10]:
for name, cm in [('LDA', cm_lda), ('QDA', cm_qda)]:
    TP, FN, FP, TN = cm[0][0], cm[0][1], cm[1][0], cm[1][1]
    print(f'{name} TPR: {round(TP / (TP + FN) * 100, 2)}%, TNR: {round(TN / (TN + FP) * 100, 2)}%')

LDA TPR: 9.09%, TNR: 94.52%
QDA TPR: 12.12%, TNR: 97.26%


## Q5 : Trading strategy vs buy-and-hold for LDA and QDA

In [11]:
test_reset = test.reset_index(drop=True).copy()
test_reset['lda_prediction'] = lda_preds

initial_balance = 100
lda_balance = initial_balance
position = None

for week_idx in range(len(test_reset)):
    current_week = test_reset.iloc[week_idx]
    year = current_week['Year']
    week_num = current_week['Week_Number']
    week_data = daily_test[(daily_test['Year'] == year) & (daily_test['Week_Number'] == week_num)]

    if len(week_data) == 0:
        continue

    if position == 'invested':
        for idx, row in week_data.iterrows():
            if pd.notna(row['Return']):
                lda_balance = lda_balance * (1 + row['Return'] / 100)

    if week_idx < len(test_reset) - 1:
        next_prediction = test_reset.iloc[week_idx + 1]['lda_prediction']
        position = 'invested' if next_prediction == 'YES' else None

test_reset['qda_prediction'] = qda_preds
qda_balance = initial_balance
position = None

for week_idx in range(len(test_reset)):
    current_week = test_reset.iloc[week_idx]
    year = current_week['Year']
    week_num = current_week['Week_Number']
    week_data = daily_test[(daily_test['Year'] == year) & (daily_test['Week_Number'] == week_num)]

    if len(week_data) == 0:
        continue

    if position == 'invested':
        for idx, row in week_data.iterrows():
            if pd.notna(row['Return']):
                qda_balance = qda_balance * (1 + row['Return'] / 100)

    if week_idx < len(test_reset) - 1:
        next_prediction = test_reset.iloc[week_idx + 1]['qda_prediction']
        position = 'invested' if next_prediction == 'YES' else None

bh_balance = initial_balance
for idx, row in daily_test.iterrows():
    if pd.notna(row['Return']):
        bh_balance = bh_balance * (1 + row['Return'] / 100)

print(f'LDA Strategy Final Balance: ${lda_balance:.2f}')
print(f'QDA Strategy Final Balance: ${qda_balance:.2f}')
print(f'Buy and Hold Final Balance: ${bh_balance:.2f}')
best = max([('LDA', lda_balance), ('QDA', qda_balance), ('Buy and Hold', bh_balance)], key=lambda x: x[1])
print(f'{best[0]} strategy results in a larger amount at the end of the testing period')

LDA Strategy Final Balance: $126.96
QDA Strategy Final Balance: $118.18
Buy and Hold Final Balance: $126.14
LDA strategy results in a larger amount at the end of the testing period
